# Phase 1a: Extract and Pivot Environmental Data

**Goal**: Extract environmental observations from database and create wide-format matrix for analysis.

**Inputs**:
- Database: `notebooks/pangenome_env_data/pangenome_env.db`
- Taxonomy: `notebooks/df_gtdb_tagged_cleaneed.tsv`

**Outputs**:
- `data/phase1_outputs/df_long.parquet` - Long-format environmental data
- `data/phase1_outputs/df_wide.parquet` - Wide-format matrix (clusters × variables)
- `data/phase1_outputs/df_taxonomy.parquet` - Taxonomy per cluster

## Setup

In [1]:
import sys
sys.path.insert(0, '../..')  # Add analysis/ to path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from analysis_lib import data_prep, utils

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
# Load configuration
config = utils.load_config('../../config.yaml')

# Key paths
DB_PATH = config['paths']['database']
TAX_PATH = config['paths']['taxonomy']
OUTPUT_DIR = config['paths']['output_dir']

print(f"Database: {DB_PATH}")
print(f"Taxonomy: {TAX_PATH}")
print(f"Output: {OUTPUT_DIR}")

Database: ../notebooks/pangenome_env_data/pangenome_env.db
Taxonomy: ../notebooks/df_gtdb_tagged_cleaneed.tsv
Output: ./data


## 1. Extract Environmental Data

Extract all environmental observations from database in long format.

In [3]:
# Extract environmental data
# Optional: filter by services if needed
services_to_include = config['services']['include']

df_long = data_prep.extract_environmental_data(
    db_path=DB_PATH,
    services=services_to_include  # Or None for all services
)

df_long.head()

OperationalError: unable to open database file

In [ ]:
# Explore data structure
print(f"Shape: {df_long.shape}")
print(f"\nColumns: {df_long.columns.tolist()}")
print(f"\nData types:\n{df_long.dtypes}")
print(f"\nMemory usage: {df_long.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# Summary statistics
print("Summary Statistics:")
print(f"  Total observations: {len(df_long):,}")
print(f"  Unique clusters: {df_long['cluster_id'].nunique():,}")
print(f"  Unique services: {df_long['service_name'].nunique()}")
print(f"  Unique variables: {df_long['variable'].nunique()}")

print(f"\nServices:")
print(df_long['service_name'].value_counts())

In [ ]:
# Save long-format checkpoint
utils.save_checkpoint(df_long, phase=1, name='df_long', output_dir=OUTPUT_DIR)

## 2. Pivot to Wide Format

Convert to wide format: clusters × variables

In [ ]:
# Pivot to wide format
df_wide = data_prep.pivot_to_wide(
    df_long,
    agg_func='mean',  # Aggregate multiple observations per cluster-variable
    add_service_prefix=True  # Create names like "NASA_POWER__temperature"
)

df_wide.head()

In [ ]:
# Inspect wide format
print(f"Shape: {df_wide.shape}")
print(f"  Clusters: {df_wide.shape[0]:,}")
print(f"  Variables: {df_wide.shape[1]}")
print(f"\nFirst 10 variables:\n{df_wide.columns[:10].tolist()}")
print(f"\nMemory usage: {df_wide.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# Data completeness
completeness = df_wide.notna().sum() / len(df_wide)

print("Data Completeness:")
print(f"  Mean: {completeness.mean():.1%}")
print(f"  Median: {completeness.median():.1%}")
print(f"  Min: {completeness.min():.1%}")
print(f"  Max: {completeness.max():.1%}")
print(f"\n  Variables with:")
print(f"    >90% data: {(completeness > 0.9).sum()}")
print(f"    >70% data: {(completeness > 0.7).sum()}")
print(f"    >50% data: {(completeness > 0.5).sum()}")
print(f"    <10% data: {(completeness < 0.1).sum()} (very sparse)")

In [ ]:
# Visualize completeness distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of completeness
completeness.hist(bins=50, ax=axes[0])
axes[0].axvline(0.7, color='red', linestyle='--', label='70% threshold')
axes[0].set_xlabel('Data Completeness')
axes[0].set_ylabel('Number of Variables')
axes[0].set_title('Distribution of Variable Completeness')
axes[0].legend()

# Top/bottom variables
top_bottom = pd.concat([
    completeness.nlargest(10),
    completeness.nsmallest(10)
])
top_bottom.plot(kind='barh', ax=axes[1])
axes[1].axvline(0.7, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Completeness')
axes[1].set_title('Most and Least Complete Variables')

plt.tight_layout()
plt.show()

In [ ]:
# Save wide-format checkpoint
utils.save_checkpoint(df_wide, phase=1, name='df_wide', output_dir=OUTPUT_DIR)

## 3. Load Taxonomy

Load taxonomic assignments for each cluster.

In [ ]:
# Load taxonomy
df_taxonomy = data_prep.load_taxonomy(
    taxonomy_file=TAX_PATH,
    min_genomes=config['selection']['min_genomes_per_cluster']
)

df_taxonomy.head()

In [ ]:
# Explore taxonomy
print(f"Shape: {df_taxonomy.shape}")
print(f"\nColumns: {df_taxonomy.columns.tolist()}")
print(f"\nTop 10 phyla:")
print(df_taxonomy['phylum'].value_counts().head(10))

In [ ]:
# Visualize phylum distribution
fig, ax = plt.subplots(figsize=(10, 6))
df_taxonomy['phylum'].value_counts().head(15).plot(kind='barh', ax=ax)
ax.set_xlabel('Number of Clusters')
ax.set_title('Top 15 Phyla by Cluster Count')
plt.tight_layout()
plt.show()

In [ ]:
# Create presence/absence matrix for phyla
df_phylum_pa = data_prep.create_presence_absence_matrix(
    df_taxonomy,
    taxonomic_level='phylum',
    min_prevalence=config['taxonomy']['min_prevalence']
)

print(f"\nPresence/absence matrix shape: {df_phylum_pa.shape}")
df_phylum_pa.head()

In [ ]:
# Save taxonomy checkpoint
utils.save_checkpoint(df_taxonomy, phase=1, name='df_taxonomy', output_dir=OUTPUT_DIR)
utils.save_checkpoint(df_phylum_pa, phase=1, name='df_phylum_pa', output_dir=OUTPUT_DIR)

## 4. Merge Environmental and Taxonomic Data

Create combined dataset for analysis.

In [ ]:
# Merge datasets
df_combined = data_prep.merge_env_taxonomy(
    df_wide,
    df_taxonomy,
    how='inner'  # Only keep clusters with both env and tax data
)

print(f"Combined shape: {df_combined.shape}")
df_combined.head()

In [ ]:
# Check alignment
print("Data Alignment:")
print(f"  Env data clusters: {len(df_wide)}")
print(f"  Tax data clusters: {len(df_taxonomy)}")
print(f"  Combined clusters: {len(df_combined)}")
print(f"  Lost in merge: {len(df_wide) + len(df_taxonomy) - 2*len(df_combined)}")

In [ ]:
# Save combined checkpoint (optional - can recreate from df_wide + df_taxonomy)
# utils.save_checkpoint(df_combined, phase=1, name='df_combined', output_dir=OUTPUT_DIR)

## Summary

✅ **Phase 1a Complete**

**Created:**
- `data/phase1_outputs/df_long.parquet` - Long-format environmental data
- `data/phase1_outputs/df_wide.parquet` - Wide-format matrix
- `data/phase1_outputs/df_taxonomy.parquet` - Taxonomy
- `data/phase1_outputs/df_phylum_pa.parquet` - Phylum presence/absence

**Next Steps:**
- `01b_missing_data.ipynb` - Analyze missing data patterns
- `01c_variable_characterization.ipynb` - Characterize variable distributions

In [ ]:
# Check all Phase 1 checkpoints
utils.print_phase_status(output_dir=OUTPUT_DIR)